# Notebook 10: Automated Causal Trace Mapping

This notebook is my implementation of **Mechanistic Localization** from my methodology.

Before I can unlearn a fact in a quantization-aware way, I first need to know *where* that fact is physically stored inside the model. Research on knowledge editing (like the ROME paper) shows that factual associations tend to be stored in specific MLP layers rather than being spread evenly across the whole model. So instead of applying my unlearning update to the entire model, I want to find the small set of layers that actually hold each fact from my forget set, and only touch those.

To do this, I use **causal tracing** (also called activation patching). The idea is simple: I run the model twice, once with a "clean" prompt (where it knows the fact) and once with a "corrupted" prompt (where the fact is hidden/broken). Then I go layer by layer and copy the clean activation into the corrupted run, and check how much of the original answer "comes back". If patching a certain layer brings the answer back strongly, that layer is important for storing that fact.

**Inputs to this notebook:**
- The fp16 target model (already saved to my Google Drive from earlier notebooks)
- `forget_set_traced.csv` — my forget set with clean/corrupted prompt pairs and the target token to check

**Output of this notebook:**
- `trace_map.json` — for every article in my forget set, this stores the top-2 most important layers plus the full 32-layer recovery scores (I keep all 32 so I can plot saliency graphs later for my thesis, not just the top-2).

This output file is what I will feed into Notebook 10b, where I actually build the Quantization-Aware Training loop and only update the neurons in the layers identified here.

In [ ]:
!pip install torch transformers pandas transformer_lens accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.6 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=c9dd000da5bf27140e1425d03f8c764d2b8c5286326087c14e1e745ae1a034b6
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


## Loading the Model with TransformerLens

A normal Hugging Face model doesn't let me easily "hook into" the internal computations of each layer while it's running. To do activation patching I need access to the intermediate activations (specifically the MLP output of each layer), so I use the `transformer_lens` library, which wraps a Hugging Face model in a `HookedTransformer` and exposes every internal activation as something I can read or overwrite during a forward pass.

One practical issue I ran into: loading the model directly to the GPU while also converting/wrapping it for TransformerLens caused memory spikes that crashed my Colab session. So here I load the base Hugging Face model to CPU first, then let `HookedTransformer.from_pretrained` move the wrapped version to the GPU, and finally delete the CPU copy to free up RAM. This is just a memory-safety workaround for Colab, not part of the actual research method.

In [ ]:
# 1. Import libraries
import torch
import pandas as pd
import json
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformer_lens import HookedTransformer
from tqdm import tqdm

# 2. Paths
MODEL_PATH = "/content/drive/MyDrive/ResearchProject/phi3-bucket-collapse/models/target_model_fp16"
TRACED_CSV_PATH = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set_traced.csv"
OUTPUT_JSON_PATH = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/trace_map.json" # "../data/processed/trace_map.json"

# 3. Load Tokenizer & Model (Memory-Safe Version)
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading Base HuggingFace Model to CPU (to prevent VRAM spike)...")
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    dtype=torch.float16,
    device_map="cpu"  # Loads to CPU RAM first to prevent Colab crashes
)

print("Wrapping model in HookedTransformer and moving to GPU...")
model = HookedTransformer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    hf_model=hf_model,
    device="cuda",    # Safely moves the formatted model to the GPU
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False
)
model.eval()

# Delete redundant CPU model to free up System RAM
del hf_model
gc.collect()
torch.cuda.empty_cache()
print("✅ Model successfully loaded to GPU.")

Loading Tokenizer...
Loading Base HuggingFace Model to CPU (to prevent VRAM spike)...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Wrapping model in HookedTransformer and moving to GPU...
Loaded pretrained model microsoft/Phi-3-mini-4k-instruct into HookedTransformer
✅ Model successfully loaded to GPU.


## Causal Tracing Algorithm

This is the core of Stage 1. For each fact in my forget set, I do the following:

1. **Clean run:** run the model on the clean prompt (the one where the correct fact is present) and record the probability it gives to the correct target token. Call this `clean_prob`.
2. **Corrupted run:** run the model on a corrupted version of the prompt (where the subject/fact is broken or replaced) and record the probability of the same target token. Call this `corrupted_prob`. Normally this should be much lower than `clean_prob`, since the model shouldn't be confident about the fact anymore.
3. **Patched run:** for each MLP layer `l`, I run the model on the corrupted prompt again, but this time I "patch in" (overwrite) that layer's MLP output activation with the one recorded from the clean run. I then check how much the target token probability recovers. Call this `patched_prob`.

I calculate a **recovery score** for each layer using:
$$
recovery(l) = (patched_prob(l) - corrupted_prob) / (clean_prob - corrupted_prob)
$$

The intuition here is: if patching in the clean activation at layer `l` brings the probability almost all the way back up to `clean_prob`, the recovery score will be close to 1, meaning that layer is very important for storing this fact. If it makes no difference, the score will be close to 0.

I do this for all 32 MLP layers of Phi-3-mini and keep the top-2 layers with the highest recovery score as my "knowledge neuron" location for that fact. I also save all 32 scores (not just the top-2) so I can later plot a full saliency curve across layers for my thesis results chapter, showing where knowledge tends to concentrate.


 My first version of this ran one article at a time, doing 34 separate tiny forward passes per article (1 clean + 1 corrupted + 32 layer-patched runs), which was slow given ~880 articles. I sped this up by processing multiple articles together in a single forward pass instead. Since batching requires every article in a batch to have the same sequence length, I first group ("bucket") articles by the length they end up at after clean/corrupted padding, and only batch articles from the same length-bucket together.

In [ ]:
import torch
import pandas as pd
import json
import gc
from collections import defaultdict
from tqdm import tqdm

MAX_BATCH_SIZE = 16

def build_padded_pair(tokenizer, clean_prompt, corrupted_prompt, pad_token_id):
    clean_tokens = tokenizer.encode(clean_prompt, return_tensors="pt")[0]
    corrupted_tokens = tokenizer.encode(corrupted_prompt, return_tensors="pt")[0]

    diff = clean_tokens.size(0) - corrupted_tokens.size(0)
    if diff > 0:
        pad_tensor = torch.full((diff,), pad_token_id)
        corrupted_tokens = torch.cat([pad_tensor, corrupted_tokens])
    elif diff < 0:
        pad_tensor = torch.full((-diff,), pad_token_id)
        clean_tokens = torch.cat([pad_tensor, clean_tokens])

    return clean_tokens, corrupted_tokens  # now the same length as each other

# Tokenize everything up front and bucket by sequence length
traced_df = pd.read_csv(TRACED_CSV_PATH)
pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

buckets = defaultdict(list)

print("Tokenizing and bucketing articles by sequence length...")
for index, row in tqdm(traced_df.iterrows(), total=len(traced_df)):
    article_id = str(row.get('id', index))
    try:
        target_token_id = tokenizer.encode(row['target_token'], add_special_tokens=False)[0]
        clean_tokens, corrupted_tokens = build_padded_pair(
            tokenizer, row['clean_prompt'], row['corrupted_prompt'], pad_token_id
        )
        L = clean_tokens.size(0)
        buckets[L].append((article_id, clean_tokens, corrupted_tokens, target_token_id, row['text']))
    except Exception as e:
        print(f"⚠️ Skipping article {article_id} during tokenization: {e}")
        continue

print(f"Grouped {len(traced_df)} articles into {len(buckets)} length-buckets.")

# Batched causal tracing
trace_map = {}
num_layers = model.cfg.n_layers

for L, items in tqdm(buckets.items(), desc="Processing length-buckets"):
    # Split large buckets into smaller sub-batches to control GPU memory
    for chunk_start in range(0, len(items), MAX_BATCH_SIZE):
        chunk = items[chunk_start:chunk_start + MAX_BATCH_SIZE]
        article_ids = [c[0] for c in chunk]
        clean_batch = torch.stack([c[1] for c in chunk]).to("cuda")        # (B, L)
        corrupted_batch = torch.stack([c[2] for c in chunk]).to("cuda")    # (B, L)
        target_ids = torch.tensor([c[3] for c in chunk], device="cuda")   # (B,)
        texts = [c[4] for c in chunk]
        B = len(chunk)

        with torch.no_grad():
            # Clean run — now processes the WHOLE batch in one forward pass
            clean_logits, clean_cache = model.run_with_cache(clean_batch)
            clean_probs = torch.softmax(clean_logits[:, -1, :], dim=-1)
            clean_probs = clean_probs.gather(1, target_ids.unsqueeze(1)).squeeze(1)  # (B,)

            # Corrupted run (baseline), also batched
            corrupted_logits = model(corrupted_batch)
            corrupted_probs = torch.softmax(corrupted_logits[:, -1, :], dim=-1)
            corrupted_probs = corrupted_probs.gather(1, target_ids.unsqueeze(1)).squeeze(1)  # (B,)

            recovery_scores = torch.zeros(B, num_layers, device="cuda")

            for layer in range(num_layers):
                hook_point = f"blocks.{layer}.hook_mlp_out"

                def patch_mlp_activation(activations, hook):
                    activations[:] = clean_cache[hook.name][:]
                    return activations

                with model.hooks(fwd_hooks=[(hook_point, patch_mlp_activation)]):
                    patched_logits = model(corrupted_batch)
                    patched_probs = torch.softmax(patched_logits[:, -1, :], dim=-1)
                    patched_probs = patched_probs.gather(1, target_ids.unsqueeze(1)).squeeze(1)  # (B,)

                recovery = (patched_probs - corrupted_probs) / (clean_probs - corrupted_probs + 1e-10)
                recovery_scores[:, layer] = recovery

        # Unpack results for each article in this chunk
        for i, article_id in enumerate(article_ids):
            scores_i = recovery_scores[i]
            top_k_values, top_k_indices = torch.topk(scores_i, k=2)
            trace_map[article_id] = {
                "text": texts[i],
                "top_layers": top_k_indices.tolist(),
                "all_layer_scores": scores_i.tolist()
            }

        del clean_cache, clean_logits, corrupted_logits, recovery_scores
        torch.cuda.empty_cache()
        gc.collect()

with open(OUTPUT_JSON_PATH, "w") as f:
    json.dump(trace_map, f, indent=4)

print(f"\n✅ Trace mapping complete! Blueprint saved to {OUTPUT_JSON_PATH}")

Tokenizing and bucketing articles by sequence length...


100%|██████████| 888/888 [00:00<00:00, 2192.80it/s]


Grouped 888 articles into 39 length-buckets.


Processing length-buckets: 100%|██████████| 39/39 [07:26<00:00, 11.45s/it]


✅ Trace mapping complete! Blueprint saved to /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/trace_map1.json


## Summary

At this point I have generated `trace_map.json`, which tells me, for every fact in my forget set, which MLP layers are most responsible for storing it (top-2 layers), along with the full layer-by-layer recovery profile for visualization later.